# 2D Ising Model: Temperature-Conditioned Diffusion

This notebook showcases a **conditional Denoising Diffusion Probabilistic Model (DDPM)** trained to generate spin configurations of the 2D Ising Model on a $64 \times 64$ square lattice, conditioned on physical temperature $T$.

## Physical Context
The 2D Ising Model is a classic model of statistical mechanics describing ferromagnetism. The Hamiltonian is:
$$H = -J \sum_{\langle i,j \rangle} s_i s_j$$
where $s_i \in \{-1, +1\}$ are spins, and $\langle i,j \rangle$ indicates nearest neighbors. In the infinite-volume limit, it exhibits a second-order phase transition at the critical temperature:
$$T_c = \frac{2}{\ln(1 + \sqrt{2})} \approx 2.269$$

*   **Ordered Phase ($T < T_c$)**: Spins align cooperatively, creating large single-domain magnetization ($|M| \to 1$).
*   **Critical Phase ($T \approx T_c$)**: Fractal-like spin clusters of all scales form (diverging correlation length $\xi$, critical fluctuations).
*   **Disordered Phase ($T > T_c$)**: Thermal fluctuations dominate; spins behave like random noise ($|M| \to 0$).

Our goal is to show that a conditional DDPM can learn this entire phase diagram and interpolate thermodynamics seamlessly.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

# Package imports
from src.model import UNet_S
from src.diffusion import Diffuser
from src.evaluate import binarize_sign, compute_all_observables

## 1. Load Pre-trained Model and Standalone EMA Weights

We load our conditional UNet architecture and restore the Exponential Moving Average (EMA) parameter checkpoint.

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate model and noise scheduler
model = UNet_S().to(device)
diffuser = Diffuser(schedule="cosine").to(device)

# Load standalone EMA weights extracted for Hugging Face
checkpoint_path = "checkpoints/Ising_DDPM_1_E100_ema_only.pt"
if os.path.exists(checkpoint_path):
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    print("Loaded standalone EMA weights successfully.")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}.")
model.eval();

## 2. Generate Spin Configurations across Phase Diagram

We generate configurations at three characteristic temperatures:
1.  **Ordered Phase** ($T = 1.0$)
2.  **Critical Phase** ($T = 2.27$)
3.  **Disordered Phase** ($T = 4.0$)

In [ ]:
n_samples = 4
target_temps = [1.0, 2.27, 4.0]
generated_samples = {}

for T in target_temps:
    print(f"Generating configurations at T = {T}...")
    # Generate batch
    with torch.no_grad():
        # sample using standard DDPM scheduler
        raw_x0 = diffuser.sample(model, n_samples=n_samples, T_phys=T)
    
    # Apply hard sign thresholding to retrieve discrete spins (+1 / -1)
    spins = binarize_sign(raw_x0.cpu().numpy().squeeze(1))
    generated_samples[T] = spins

## 3. Visualize Generated Configurations

Let's look at the generated states. Notice how domains form at $T=1.0$, fractal boundary paths appear at $T=2.27$, and noise dominates at $T=4.0$.

In [ ]:
fig, axes = plt.subplots(3, n_samples, figsize=(12, 9))

for i, T in enumerate(target_temps):
    spins = generated_samples[T]
    for j in range(n_samples):
        ax = axes[i, j]
        ax.imshow(spins[j], cmap="binary", interpolation="nearest")
        ax.set_xticks([])
        ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(f"T = {T}", fontsize=14, fontweight="bold")

plt.suptitle("Generated 2D Ising Configurations (64x64)", fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

## 4. Quantitative Thermodynamics: Diffusion vs Monte Carlo

We load a set of generated samples and compare the computed physical observables (Energy $E$, Magnetization $|M|$, Specific Heat $C_v$, Susceptibility $\chi$, Binder Cumulant $U_4$, and Correlation Length $\xi$) against the Monte Carlo benchmark dataset (`data/ising_benchmark.npz`).

In [ ]:
# Load benchmark dataset
benchmark = np.load("data/ising_benchmark.npz")
mc_configs = benchmark["configs"]          # (n_T, n_samples, L, L)
mc_temps = benchmark["temperatures"]      # (n_T,)

print(f"Loaded Monte Carlo benchmark data with {len(mc_temps)} temperatures.")

# Compute MC observables
mc_obs = {}
for idx, T in enumerate(mc_temps):
    mc_obs[T] = compute_all_observables(mc_configs[idx], T)

# Generate configuration arrays for evaluation
# To make execution fast in the notebook, we generate 50 samples per temperature
n_eval_samples = 50
gen_obs = {}

print(f"Generating {n_eval_samples} evaluation configs per temperature across the grid...")
for T in mc_temps:
    with torch.no_grad():
        # Use fast respaced sampler (100 steps) for quick evaluation in notebook
        raw_x0 = diffuser.sample_respaced(model, n_samples=n_eval_samples, T_phys=T, n_steps=100)
    
    spins = binarize_sign(raw_x0.cpu().numpy().squeeze(1))
    gen_obs[T] = compute_all_observables(spins, T)

## 5. Plot Observables Curves

We plot the physical quantities to show how the diffusion model matches the ground truth thermodynamics.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Thermodynamic Observables: Diffusion Model vs Monte Carlo Ground Truth", fontsize=15, y=0.98)

quantities = [
    ("E", "Energy per spin $E/N$"),
    ("M", "Magnetization $|M|/N$"),
    ("C_v", "Specific heat $C_v$"),
    ("chi", r"Susceptibility $\chi$"),
    ("U_4", "Binder cumulant $U_4$"),
    ("xi", r"Correlation length $\xi$"),
]

for ax, (key, ylabel) in zip(axes.flat, quantities):
    mc_vals = [mc_obs[T][key] for T in mc_temps]
    gen_vals = [gen_obs[T][key] for T in mc_temps]
    
    ax.plot(mc_temps, mc_vals, "o-", label="MC Ground Truth", color="teal", alpha=0.8)
    ax.plot(mc_temps, gen_vals, "s--", label="Model (DDPM-100)", color="tomato", alpha=0.8)
    ax.axvline(2.269, color="gray", linestyle=":", label="$T_c \approx 2.27$")
    ax.set_xlabel("Temperature $T$")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()